In [ ]:
!pip uninstall -y gradio transformers
!pip install gradio transformers --upgrade
!pip install --upgrade reportlab

Found existing installation: gradio 6.8.0
Uninstalling gradio-6.8.0:
  Successfully uninstalled gradio-6.8.0
Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0
  Using cached gradio-6.8.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
Using cached gradio-6.8.0-py3-none-any.whl (24.2 MB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)


In [ ]:
import os
os.listdir("/content")

['.config', '.gradio', 'sample_data']

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Copy_of_Docgenie7.ipynb to Copy_of_Docgenie7.ipynb


In [ ]:
import json

with open("/content/Copy_of_Docgenie7.ipynb","r") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata",{}):
    del nb["metadata"]["widgets"]

with open("/content/notebook_fixed.ipynb","w") as f:
    json.dump(nb,f)

print("Widget metadata removed successfully")

Widget metadata removed successfully


In [ ]:
import ast
import gradio as gr
from transformers import pipeline
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
import datetime
import os

In [ ]:
!pip install --upgrade transformers

from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

result = generator("What is AI?", max_length=50)
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

[{'generated_text': 'What is AI?method or technique'}]


In [ ]:
class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                return node
        return None

    @staticmethod
    def analyze_function_logic(func_node):
        analysis = {
            "has_loops": False,
            "has_conditions": False,
            "has_return": False
        }

        for node in ast.walk(func_node):
            if isinstance(node, (ast.For, ast.While)):
                analysis["has_loops"] = True
            if isinstance(node, ast.If):
                analysis["has_conditions"] = True
            if isinstance(node, ast.Return):
                analysis["has_return"] = True

        return analysis

In [ ]:
def generate_ai_docstring(code, style):
    prompt = f"""
Generate a professional {style} style Python docstring
for the following function:

{code}

Docstring:
"""

    result = generator(prompt, max_length=300, do_sample=False)
    return result[0]["generated_text"]

In [ ]:
def process_code(code, style):
    try:
        func_node = DocGenieAnalyzer.extract_function_signature(code)
        if not func_node:
            return "No function detected."

        analysis = DocGenieAnalyzer.analyze_function_logic(func_node)

        docstring = generate_ai_docstring(code, style)

        lines = code.split("\n")
        indent = " " * 4
        formatted_doc = indent + docstring.replace("\n", "\n" + indent)

        lines.insert(1, formatted_doc)

        return "\n".join(lines)

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
def generate_pdf(output_code):
    filename = "DocGenie_Report.pdf"
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph("<b>Doc-Genie Documentation Report</b>", styles["Title"]))
    elements.append(Spacer(1, 12))
    elements.append(Preformatted(output_code, styles["Code"]))
    elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"Generated on: {datetime.datetime.now()}", styles["Normal"]))

    doc.build(elements)

    return filename

In [ ]:
with gr.Blocks(title="Doc-Genie AI") as demo:

    gr.Markdown("# 📘 Doc-Genie\nAI Powered Python Docstring Generator")

    with gr.Row():
        with gr.Column():
            code_input = gr.Code(label="Enter Python Function", language="python")
            style_selector = gr.Radio(
                ["Google", "NumPy"],
                value="Google",
                label="Docstring Style"
            )
            generate_btn = gr.Button("Generate Docstring")

        with gr.Column():
            output_code = gr.Code(label="Output", language="python")
            pdf_btn = gr.Button("Download PDF")
            pdf_output = gr.File()

    generate_btn.click(
        process_code,
        inputs=[code_input, style_selector],
        outputs=output_code
    )

    pdf_btn.click(
        generate_pdf,
        inputs=output_code,
        outputs=pdf_output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c37e17786298399371.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def calculate_factorial(n: int) -> int:
    if n == 0:
        return 1
    return n * calculate_factorial(n-1)